In [1]:
import sqlite3
import numpy as np
import pandas as pd

In [5]:
rng = np.random.default_rng(42)
n = 1000

clientes = pd.DataFrame({
    "clientes_id" : np.arange(1,n+1),
    "cuota": rng.choice([15, 30, 50, 80], n),
    "antiguedad": rng.integers(1, 49, n)
})

registros = []

for cliente in clientes.itertuples():
    actividad_base = rng.uniform(8, 40)
    deterioro = rng.uniform(0, 0.18)

    for mes in range(1,7):
      actividad = max(
            0,
            actividad_base * (1 - deterioro)**(mes-1)
            + rng.normal(0, 2)
        )

      registros.append({
            "cliente_id": cliente.clientes_id,
            "mes": mes,
            "actividad": actividad
        })
actividad = pd.DataFrame(registros)

conn = sqlite3.connect("clientes.db")
clientes.to_sql("clientes", conn, if_exists="replace", index=False)
actividad.to_sql("actividad", conn, if_exists="replace", index=False)

6000

In [7]:
query = """
SELECT
    cliente_id,
    AVG(CASE WHEN mes IN (1, 2)
        THEN actividad END) AS actividad_inicial,
    AVG(CASE WHEN mes IN (5, 6)
        THEN actividad END) AS actividad_final
FROM actividad
GROUP BY cliente_id;
"""

pd.read_sql_query(query, conn)

,cliente_id,actividad_inicial,actividad_final
0,1,9.325976,6.402093
1,2,28.112251,19.870375
2,3,26.665420,14.557273
3,4,38.262757,28.460225
4,5,19.026585,13.301446
...,...,...,...
995,996,20.560952,12.178158
996,997,32.952056,27.118969
997,998,24.094458,13.722778
998,999,34.917476,19.128626


Definimos un cliente de riesgo cuando su actividad ha descendido un 30%

In [9]:
def calcular_variacion(i):
    # Filtramos la actividad para el cliente 'i' usando el DataFrame global 'actividad'
    datos_cliente = actividad[actividad['cliente_id'] == i]

    # Obtenemos el promedio de actividad de los meses 1 y 2
    inicial = datos_cliente[datos_cliente['mes'].isin([1, 2])]['actividad'].mean()

    # Obtenemos el promedio de actividad de los meses 5 y 6
    final = datos_cliente[datos_cliente['mes'].isin([5, 6])]['actividad'].mean()

    # Retornamos la variación
    return (final - inicial) / inicial

### Análisis de Impacto Económico

Supón que contactar a un cliente cuesta **3 €** y que una intervención exitosa consigue conservar **tres meses adicionales** de suscripción.

Si $p_i$ representa la probabilidad estimada de que la intervención evite su abandono, el beneficio esperado ($V_i$) sería:

$$V_i = 3 \cdot q_i \cdot p_i - 3$$

donde $q_i$ es su cuota mensual.

Como no tenemos datos de abandonos ni resultados de intervenciones, no puedes estimar $p_i$ de forma fiable. Utiliza **$p_i = 0.10$** como hipótesis ilustrativa para estudiar la sensibilidad económica.

#### Tareas:
1. Ordena los clientes de riesgo alto por $V_i$ y selecciona los **50 con mayor valor positivo**.
2. Después, repite el análisis con **$p = 0.05$** y **$p = 0.20$**. Observa cómo cambia la rentabilidad estimada de las intervenciones.

In [17]:
# 1. Primero, aseguramos cargar correctamente el DataFrame de actividad desde SQLite
actividad = pd.read_sql_query("SELECT * FROM actividad;", conn)

# 2. Creamos una copia del DataFrame de clientes para mantener los datos limpios
clientes_analisis = clientes.copy()

# 3. Aplicamos la función 'calcular_variacion' a cada 'clientes_id'
# El método .apply() pasa cada ID a tu función y guarda el resultado en una nueva columna
clientes_analisis['variacion'] = clientes_analisis['clientes_id'].apply(calcular_variacion)

# 4. Mostramos los primeros 10 resultados para validar
clientes_riesgo = clientes_analisis[
    clientes_analisis["variacion"] <= -0.30
].copy()
display(clientes_riesgo.head(10))

,clientes_id,cuota,antiguedad,variacion
0,1,15,39,-0.313520
2,3,50,27,-0.454077
4,5,30,7,-0.300902
5,6,80,45,-0.481259
8,9,15,36,-0.395423
10,11,50,13,-0.438629
11,12,80,39,-0.345458
16,17,50,19,-0.314665
18,19,80,2,-0.409973
21,22,30,4,-0.468388


In [19]:
def valor_esperado(cuota, p, meses=3, coste_contacto=3):
    return cuota * p * meses - coste_contacto

In [20]:
# 1. Creamos una copia del DataFrame de clientes de riesgo para mantener los datos limpios
clientes_probabilidad = clientes_riesgo.copy()

# 2. Definimos una probabilidad ilustrativa de p = 0.10 (10%) y coste de contacto = 3
p_ilustrativa = 0.10
coste_contacto = 3

# 3. Aplicamos la nueva función 'valor_esperado' a cada fila usando su 'cuota'
clientes_probabilidad['Valor Esperado (V_i)'] = clientes_probabilidad['cuota'].apply(
    lambda cuota: valor_esperado(cuota, p=p_ilustrativa, coste_contacto=coste_contacto)
)

# 4. Filtramos los clientes con valor esperado estrictamente positivo, ordenamos de mayor a menor y seleccionamos los 50 mejores
top_50 = (
    clientes_probabilidad[clientes_probabilidad['Valor Esperado (V_i)'] > 0]
    .sort_values('Valor Esperado (V_i)', ascending=False)
    .head(50)
)

# 5. Mostramos los primeros 10 del top 50 obtenido para validar
display(top_50.head(10))

,clientes_id,cuota,antiguedad,variacion,Valor Esperado (V_i)
11,12,80,39,-0.345458,21.0
998,999,80,14,-0.452176,21.0
528,529,80,1,-0.334252,21.0
498,499,80,23,-0.587515,21.0
502,503,80,11,-0.419132,21.0
510,511,80,17,-0.483384,21.0
513,514,80,16,-0.431916,21.0
514,515,80,12,-0.327812,21.0
552,553,80,26,-0.503660,21.0
557,558,80,38,-0.554774,21.0


In [21]:
resumen = actividad.groupby("mes")["actividad"].agg(
    ["mean", "median", "std", "min", "max"]
)

resumen["cv"] = resumen["std"] / resumen["mean"]

print(resumen)

          mean     median       std       min        max        cv
mes                                                               
1    24.089557  23.854356  9.686908  3.538014  44.030037  0.402121
2    21.985449  21.630247  8.921300  1.853495  42.530748  0.405782
3    20.075139  20.033515  8.304421  3.853867  39.366140  0.413667
4    18.423680  17.983570  8.166541  0.650940  39.159084  0.443263
5    16.894776  16.224920  7.982453  0.961260  41.896313  0.472481
6    15.724090  14.629872  7.868774  0.000000  45.003274  0.500428


In [22]:
from scipy.stats import ttest_rel

inicial = (
    actividad[actividad.mes.isin([1, 2])]
    .groupby("cliente_id")["actividad"]
    .mean()
)

final = (
    actividad[actividad.mes.isin([5, 6])]
    .groupby("cliente_id")["actividad"]
    .mean()
)

resultado = ttest_rel(
    final,
    inicial,
    alternative="less"
)

print("Estadístico:", resultado.statistic)
print("P-valor:", resultado.pvalue)

Estadístico: -42.157110604889816
P-valor: 2.9869372035934465e-224


### Contraste de hipótesis: análisis de la disminución de actividad

Una vez realizado el análisis descriptivo de los datos, queremos determinar si la disminución de actividad observada entre los primeros y los últimos meses es estadísticamente significativa.

Para ello, utilizamos un contraste **t de Student para muestras emparejadas** (*paired t-test*). Este procedimiento resulta adecuado porque comparamos dos mediciones correspondientes a los mismos clientes: la actividad media de los meses 1 y 2 frente a la actividad media de los meses 5 y 6.

Definimos, para cada cliente $i$, la diferencia de actividad:

$$D_i = A_{i,\text{final}} - A_{i,\text{inicial}}$$

Planteamos las siguientes hipótesis sobre la media poblacional de las diferencias ($\mu_D$):

$$H_0: \mu_D = 0$$
$$H_1: \mu_D < 0$$

La hipótesis nula ($H_0$) establece que no existe una disminución media de actividad en la población, mientras que la hipótesis alternativa ($H_1$) sostiene que la actividad media final es inferior a la inicial.

Fijamos un nivel de significación del 5 % ($\alpha = 0.05$) y obtenemos los siguientes resultados:

* **Estadístico $t$:** $-42.1571$
* **P-valor:** $2.9869 \times 10^{-224}$

El estadístico $t$ notablemente negativo indica que la diferencia media observada es menor que cero (la actividad final es significativamente inferior a la inicial).

Por otro lado, el p-valor es extremadamente cercano a cero, situándose muy por debajo del nivel de significación establecido. Por tanto, **rechazamos la hipótesis nula ($H_0$)**.

Este resultado proporciona evidencia estadística sumamente sólida de que existe una disminución real de la actividad media de los clientes durante el periodo analizado.

---

#### Interpretación y limitaciones

* **Actividad vs. Abandono (*Churn*):** Es fundamental distinguir entre una disminución de la actividad y una tasa de abandono de clientes. Este contraste demuestra que los clientes reducen su uso del servicio, pero no implica directamente que hayan cancelado sus suscripciones.
* **Definición de P-valor:** Recordemos que el p-valor no representa la probabilidad de que la hipótesis nula sea verdadera, sino la probabilidad de obtener un resultado tan extremo o más que el observado bajo el supuesto de que $H_0$ es cierta.

En conclusión, los resultados justifican plenamente la necesidad de realizar campañas o intervenciones dirigidas a los clientes identificados con alta pérdida de actividad.

In [23]:
import numpy as np
from scipy.stats import t

# 1. Diferencias individuales
diferencias = final - inicial

# 2. Estadísticos muestrales
n = len(diferencias)
media = np.mean(diferencias)
desviacion = np.std(diferencias, ddof=1)

# 3. Error estándar
error_estandar = desviacion / np.sqrt(n)

# 4. Cuantil de Student
t_critico = t.ppf(0.975, df=n-1)

# 5. Intervalo de confianza
limite_inferior = media - t_critico * error_estandar
limite_superior = media + t_critico * error_estandar

print("Media:", media)
print("Desviación típica:", desviacion)
print("Error estándar:", error_estandar)
print("t crítico:", t_critico)

print("IC 95%:")
print(limite_inferior, limite_superior)

Media: -6.728069774996283
Desviación típica: 5.046841313421757
Error estándar: 0.15959513539848463
t crítico: 1.9623414611334487
IC 95%:
-7.041249926183935 -6.41488962380863


### Estimación mediante un intervalo de confianza

Tras realizar el contraste de hipótesis, calculamos un intervalo de confianza del 95 % para la diferencia media de actividad entre el periodo inicial y el final.

Definimos las diferencias individuales de actividad para cada cliente como:

$$D_i = A_{i,\text{final}} - A_{i,\text{inicial}}$$

La media muestral obtenida es:

$$\bar{D} = -6.7281$$

La desviación típica de las diferencias es $s_D = 5.0468$ y el error estándar de la media es:

$$\text{SE} = \frac{s_D}{\sqrt{n}} = 0.1596$$

Como la desviación típica poblacional es desconocida, utilizamos la distribución $t$ de Student con $999$ grados de libertad ($n - 1$). El cuantil crítico para un intervalo bilateral del 95 % es $t_{0.975, 999} = 1.9623$.

Aplicando la fórmula del intervalo de confianza:

$$IC_{95\%} = \bar{D} \pm t_{0.975, 999} \frac{s_D}{\sqrt{n}}$$

Obtenemos el siguiente resultado:

$$\boxed{IC_{95\%} = [-7.0412, -6.4149]}$$

#### Interpretación de los resultados

* **Magnitud del impacto:** El intervalo indica que la disminución media de actividad en la población se estima, con un 95 % de confianza, entre $6.4149$ y $7.0412$ unidades.
* **Coherencia estadística:** Dado que el intervalo se encuentra completamente por debajo de cero, los resultados son totalmente coherentes con el contraste de hipótesis anterior, donde rechazamos la hipótesis nula de ausencia de cambio.

En consecuencia, confirmamos de forma robusta un deterioro generalizado en la actividad de los clientes. Este hallazgo subraya la importancia de identificar a los clientes en riesgo para diseñar estrategias de retención efectivas.